# Fase 2: Modelado Analítico y Chat Interactivo de Negocio
Este notebook muestra cómo consumimos el resultado estructural del modelo de LightGBM, finalizando con un panel interactivo que conecta al pipeline expuesto en FastAPI.

In [ ]:
import sys
import os

# Hot-Reloading de la arquitectura base
root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root_path not in sys.path:
    sys.path.append(root_path)

%load_ext autoreload
%autoreload 2

from src.train_mora import MoraModelTrainer
from src.config import settings
import pandas as pd

## 2.1 Simulando el Pipeline MLOps
En lugar de entrenar línea por línea ensuciando el cuaderno, llamamos a la clase compilada.

In [ ]:
# Ojo: Asegúrate de haber consolidado tu tabla analítica 'abt.csv' primero.
# abt_path = settings.DATA_PROCESSED_DIR / "abt.csv"
# model_path = settings.MODELS_DIR / "modelo_mora.pkl"
# trainer = MoraModelTrainer(abt_path, model_path)
# trainer.run_pipeline()
print(" Pipeline LightGBM Disponible vía el orquestador en src.train_mora")

## 2.2 Chat Interactivo de Negocio (Consumo REST a FastAPI)
**Nota:** El contenedor Docker uvicorn/fastapi (`docker-compose up`) debe estar levantado en el puerto `8000`. Modifica la pregunta y vuelve a correr la celda cuantas veces desees.

In [ ]:
import requests
import json
from IPython.display import display, Markdown

# Interfaz simplificada
def consultar_experto_llm(pregunta: str) -> None:
    """Consume el endpoint POST de FastAPI de forma limpia y renderiza en Markdown."""
    url = "http://localhost:8000/ask-analyst"
    payload = {"question": pregunta}
    
    try:
        response = requests.post(url, json=payload, timeout=30.0)
        response.raise_for_status()
        data_json = response.json()
        
        display(Markdown(f"**Usuario:** {data_json['question']}"))
        display(Markdown(f"**🤖 TUMIPAY LLM Expert:**\n{data_json['respuesta']}"))
        
    except requests.exceptions.ConnectionError:
        display(Markdown("**❌ Error:** No se pudo conectar a la API. ¿Está levantado el docker-compose o Uvicorn en el puerto 8000?"))
    except Exception as e:
        display(Markdown(f"**❌ Error inesperado:** {e}"))

# ====================================================
# PREGUNTA INTERACTIVA (Ejecutar esta celda libremente)
# ====================================================
mi_pregunta = "¿Cuáles son los segmentos de mayor riesgo según tu contexto y cómo mitigamos el Leakage en el Pipeline?"

consultar_experto_llm(mi_pregunta)